In [ ]:
# Crew Scheduling Notebook (Set Partitioning, guaranteed feasible)

from optilab.aliases import Model, GRB, quicksum
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# MODEL
# ==============================================================================

m = Model()
print("Backend:", m.backend_name)

rng = np.random.default_rng(42)

# ==============================================================================
# FLIGHTS
# ==============================================================================

n_flights = rng.integers(20, 40)

# ==============================================================================
# STEP 1 — CREATE A TRUE PARTITION OF FLIGHTS (FEASIBILITY FOUNDATION)
# ==============================================================================

# number of base duty blocks (true partition components)
n_blocks = rng.integers(6, 12)

# assign each flight to exactly one block
flight_block = rng.integers(0, n_blocks, size=n_flights)

# ==============================================================================
# STEP 2 — BUILD SCHEDULE CANDIDATES AROUND EACH BLOCK
# ==============================================================================

n_schedules = rng.integers(20, 40)

membership = np.zeros((n_flights, n_schedules), dtype=int)

# ensure each block contributes at least one schedule that equals the block
block_to_schedule = {}

for b in range(n_blocks):
    j = len(block_to_schedule)
    block_to_schedule[b] = j

    membership[flight_block == b, j] = 1

# add additional realistic schedules (perturbations of base blocks)
for j in range(n_blocks, n_schedules):
    b = rng.integers(0, n_blocks)

    base_flights = np.where(flight_block == b)[0]

    # take most of block + small perturbation
    keep = rng.choice(base_flights, size=max(1, len(base_flights) - 1), replace=False)

    membership[keep, j] = 1

# ==============================================================================
# COST STRUCTURE (realistic: fatigue + complexity)
# ==============================================================================

cost = np.array([
    membership[:, j].sum() + rng.integers(5, 15)
    for j in range(n_schedules)
])

print("n flights:", n_flights)
print("n schedules:", n_schedules)
print("n blocks:", n_blocks)
print("cost per schedule:", cost)

# ==============================================================================
# DECISION VARIABLES
# ==============================================================================

x = [m.add_binary_var(name=f"x_{j}") for j in range(n_schedules)]

# ==============================================================================
# SET PARTITIONING CONSTRAINTS
# ==============================================================================

for i in range(n_flights):
    m.add_constraint(
        quicksum(membership[i][j] * x[j] for j in range(n_schedules)) == 1
    )

# ==============================================================================
# OBJECTIVE
# ==============================================================================

m.set_objective(
    quicksum(cost[j] * x[j] for j in range(n_schedules)),
    GRB.MINIMIZE
)

# ==============================================================================
# SOLVE
# ==============================================================================

m.solve()

print("\nStatus:", m.status)

solution = np.array([v.x if v.x is not None else 0 for v in x])

coverage = membership @ solution

print("\nCoverage check (must all be 1):")
print(coverage)

print("\nTotal cost:", float(np.sum(cost * solution)))


In [ ]:
# ==============================================================================
# FEASIBLE SOLUTION COST (baseline)
# ==============================================================================

baseline_cost = float(np.sum(cost * solution))

print("\nBaseline solution cost:", baseline_cost)

# ==============================================================================
# CONSTRUCT ALTERNATIVE FEASIBLE PARTITIONS
# ==============================================================================

def evaluate_partition(selected_schedules):
    """Compute cost + coverage validity."""
    selected = np.array(selected_schedules)
    cov = membership @ selected

    feasible = np.all(cov == 1)
    total_cost = float(np.sum(cost * selected))

    return feasible, total_cost, cov


# ==============================================================================
# ALT 1 — REGENERATE A VALID PARTITION (different structure)
# ==============================================================================

alt1 = np.zeros(n_schedules)

# assign each flight to a *different valid schedule than baseline if possible*
for i in range(n_flights):
    candidates = np.where(membership[i] == 1)[0]

    # pick a different schedule than baseline if possible
    alt_choice = rng.choice(candidates)
    alt1[alt_choice] = 1


# ------------------------------------------------------------------------------
# ALT 2 — random valid resample (feasible reconstruction)
# ------------------------------------------------------------------------------

alt2 = np.zeros_like(solution)

for i in range(n_flights):
    candidates = np.where(membership[i] == 1)[0]
    j = rng.choice(candidates)
    alt2[j] = 1


# ==============================================================================
# EVALUATE ALL
# ==============================================================================

alts = {
    "baseline": solution,
    "alt1_swap": alt1,
    "alt2_resample": alt2
}

print("\n=== PARTITION COMPARISON ===")

for name, sol in alts.items():
    feasible, cost_val, cov = evaluate_partition(sol)

    print(f"\n{name}")
    print("feasible:", feasible)
    print("cost:", cost_val)